# Online Shopper Purchase Intention — Supervised & Unsupervised ML

**Author:** Tony James  
**Dataset:** Online Shoppers Purchasing Intention Dataset (UCI Machine Learning Repository)  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn

---

## Project Overview

E-commerce platforms generate vast amounts of session data — page views, time on site, bounce rates, traffic sources — but most visits don't end in a purchase. This project tackles two complementary questions:

1. **Can we predict whether a browsing session will result in a purchase?** *(Supervised — Logistic Regression)*
2. **What natural groups exist within shopper behaviour?** *(Unsupervised — K-Means Clustering)*

**Business value:** Predicting purchase intent enables real-time intervention — targeted offers, personalised recommendations, or live chat prompts. Clustering reveals distinct shopper profiles that can shape longer-term marketing strategy.

### Dataset
- **Source:** UCI Machine Learning Repository — Sakar et al. (2019)
- **Size:** 12,330 browsing sessions
- **Features:** 17 (mix of numeric and categorical — page visit counts, durations, bounce/exit rates, traffic type, visitor type, month, weekend flag)
- **Target:** `Revenue` — True if session ended in a purchase, False otherwise
- **Class imbalance:** ~84% non-purchase, ~16% purchase sessions


## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4" 

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt



## 2. Load Dataset

Each row represents one browsing session. The target variable `Revenue` indicates whether that session resulted in a purchase.


In [ ]:

# -----------------------------
# [2] Load Dataset
# -----------------------------
df = pd.read_csv("online_shoppers_intention.csv")

In [ ]:
print("Shape:", df.shape)

In [ ]:
display(df.head())


In [ ]:
print(df.dtypes)


## 3. Feature Engineering & Preprocessing

The dataset contains 14 numeric features and 3 categorical features. Key preprocessing steps:

- **StandardScaler** applied to numeric features — ensures distance-based models aren't dominated by large-scale variables like `ProductRelated_Duration`
- **OneHotEncoder** applied to categorical features (`Month`, `VisitorType`, `Weekend`) — converts labels to binary flags
- A **ColumnTransformer Pipeline** ensures the same preprocessing is applied consistently to both train and test sets, preventing data leakage
- **Class imbalance** handled by setting `class_weight='balanced'` in Logistic Regression — purchase sessions (16%) are underrepresented and need upweighting


In [ ]:
# -----------------------------
# [3] Basic Data Checks (EDA-lite)
# -----------------------------
# Missing values
missing = df.isna().sum()
print("Missing values (top):",missing)

In [ ]:
# Target distribution
TARGET_COL = "Revenue"  
print("\nTarget value counts:")
display(df[TARGET_COL].value_counts(dropna=False))

In [ ]:
# -----------------------------
# [4] Define Features/Target
# -----------------------------

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# If y is boolean True/False, convert to int 0/1 for convenience

if y.dtype == "bool":
    y = y.astype(int)

# Identify column types
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 4. Supervised Model — Logistic Regression

**Why Logistic Regression?** It's interpretable, works well for binary classification, and is a strong baseline for understanding which features drive purchase intent. It outputs probability scores which allow flexible threshold tuning via ROC curves.

The model is trained on 80% of sessions and evaluated on the held-out 20%.


In [ ]:
# -----------------------------
# [5] Preprocessing Pipeline
# -----------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



RANDOM_STATE = 42



numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)


In [ ]:
# -----------------------------
# [6] Train/Test Split (Supervised)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y if y.nunique() == 2 else None
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)



In [ ]:
# -----------------------------
# [7] Supervised Model: Logistic Regression
# -----------------------------
# Choose solver that works well for many features after one-hot encoding.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)


log_reg = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE,
    class_weight="balanced"  
)

clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", log_reg)
])

# Train
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)
# For ROC-AUC 
if hasattr(clf.named_steps["model"], "predict_proba"):
    y_proba = clf.predict_proba(X_test)[:, 1]
else:
    y_proba = None

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n--- Logistic Regression Evaluation ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

if y_proba is not None and y_test.nunique() == 2:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC  : {auc:.4f}")


In [ ]:
# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

## 5. Model Evaluation

### Metrics used
Given the class imbalance (~84% non-purchase), accuracy alone is misleading. We use:

| Metric | What it tells us |
|---|---|
| **Accuracy** | Overall correctness |
| **Precision** | Of predicted purchases, how many were real? |
| **Recall** | Of actual purchases, how many did we catch? |
| **F1-Score** | Balance between precision and recall |
| **ROC-AUC** | Discrimination ability across all thresholds |

### Results
- Accuracy: **0.85** — strong overall
- Recall: **0.75** — catches 75% of actual buyers
- ROC-AUC: **0.90** — excellent separation between buyers and non-buyers

The confusion matrix and ROC curve below show the model's performance across both classes.


In [ ]:
# ROC curve plot 
if y_proba is not None and y_test.nunique() == 2:
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("ROC Curve - Logistic Regression")
    plt.show()


In [ ]:
# -----------------------------
# [8] Interpretability: Feature Effects (Optional but good for report)
# -----------------------------
# Get feature names after preprocessing and align with coefficients

# Fit preprocessor on full training data to get feature names
preprocessor_fitted = clf.named_steps["preprocess"]
feature_names = []

# Numeric names
feature_names.extend(numeric_features)

# Categorical one-hot names
ohe = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
feature_names.extend(cat_feature_names)

coefs = clf.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df_sorted = coef_df.sort_values("abs_coef", ascending=False)

print("\nTop 15 strongest coefficients (by absolute magnitude):")
display(coef_df_sorted.head(15)[["feature", "coefficient"]])


In [ ]:
# -----------------------------
# [9] Unsupervised Prep: Build Feature Matrix for K-Means
# -----------------------------
# We’ll preprocess ALL X (same preprocessor) to get a numeric matrix.

X_all_processed = preprocessor.fit_transform(X)

print("Processed feature matrix shape for clustering:", X_all_processed.shape)

## 6. Unsupervised Model — K-Means Clustering

**Why K-Means?** While Logistic Regression tells us *whether* someone will buy, K-Means reveals *who* our shoppers are — without any labels. Grouping sessions by behavioural similarity uncovers natural shopper profiles that prediction alone misses.

**Process:**
1. Apply the same preprocessing pipeline to the full dataset (excluding `Revenue` to avoid bias)
2. Test k = 2 to 10 clusters using the **Elbow Method** (inertia) and **Silhouette Score**
3. Select the optimal k and fit the final model
4. Profile each cluster by behavioural averages and purchase rates


In [ ]:
# -----------------------------
# [10] K-Means: Elbow Method (Inertia) + Silhouette Score
# -----------------------------
# Typical k range: 2 to 10 (or higher). Keep reasonable for interpretation.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 11)

inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_all_processed)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_all_processed, labels))



In [ ]:
# Elbow plot
plt.figure()
plt.plot(list(k_range), inertias, marker="o")
plt.title("Elbow Method (Inertia) for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.show()

In [ ]:
# Silhouette plot
plt.figure()
plt.plot(list(k_range), silhouettes, marker="o")
plt.title("Silhouette Score for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()

# Choose best k by silhouette (common approach) OR pick at elbow point for interpretability
best_k = int(list(k_range)[int(np.argmax(silhouettes))])
print("Best k by silhouette:", best_k)

In [ ]:
# -----------------------------
# [11] Fit Final K-Means & Assign Cluster Labels
# -----------------------------
FINAL_K = best_k  # you can override after inspecting plots
kmeans_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans_final.fit_predict(X_all_processed)

df_clustered = df.copy()
df_clustered["Cluster"] = clusters

print("\nCluster counts:")
display(df_clustered["Cluster"].value_counts().sort_index())



In [ ]:
# -----------------------------
# [12] Cluster Profiling
# -----------------------------
# Compare clusters using numeric feature means
cluster_profile_num = df_clustered.groupby("Cluster")[numeric_features].mean()
print("\nNumeric feature means by cluster:")
display(cluster_profile_num)

# Compare categorical distributions (example for a few key categorical columns)
for col in categorical_features[:5]:  # adjust as needed
    print(f"\nDistribution of {col} by Cluster:")
    display(pd.crosstab(df_clustered["Cluster"], df_clustered[col], normalize="index"))

## 7. Cluster Analysis & Business Insights

After fitting K-Means with the optimal k, each session receives a cluster label. We then compare clusters across:
- Numeric feature means (page counts, time on site, bounce rates)
- Categorical distributions (visitor type, traffic source, month)
- **Purchase rate per cluster** — this connects unsupervised findings back to business outcomes

### Key findings from clustering
- **Cluster 0** (high-intent shoppers): Higher page values, lower bounce rates, purchase rate ~28%
- **Cluster 1** (browsers): Longer sessions but lower engagement quality, purchase rate ~13%

This segmentation enables targeted interventions — Cluster 0 visitors are close to converting and may respond well to a well-timed discount or product recommendation.


In [ ]:
# Revenue rate by cluster (helps connect clustering to business outcome)
# If Revenue is boolean or 0/1
if df_clustered[TARGET_COL].dtype in [np.int64, np.int32, np.float64, np.float32, "bool"]:
    revenue_rate = df_clustered.groupby("Cluster")[TARGET_COL].mean().sort_values(ascending=False)
    print("\nRevenue (purchase) rate by cluster:")
    display(revenue_rate)

In [ ]:
# -----------------------------
# [13] Final Summary (Display Only – No File Saving)
# -----------------------------

print("\n================ FINAL MODEL SUMMARY ================\n")

# Supervised model metrics
print("Supervised Model: Logistic Regression")
print("------------------------------------")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-Score      : {f1:.4f}")

if "auc" in locals() and auc is not None:
    print(f"ROC-AUC       : {auc:.4f}")

print("\n")

# Unsupervised model summary
print("Unsupervised Model: K-Means Clustering")
print("-------------------------------------")
print(f"Final number of clusters (k): {FINAL_K}")
print(f"Best k by Silhouette Score : {best_k}")
print(f"Maximum Silhouette Score  : {np.max(silhouettes):.4f}")

print("\nCluster sizes:")
display(df_clustered["Cluster"].value_counts().sort_index())

# Revenue rate by cluster (business insight)
if df_clustered[TARGET_COL].dtype in [np.int64, np.int32, np.float64, np.float32, "bool"]:
    print("\nAverage Revenue (Purchase Rate) by Cluster:")
    display(
        df_clustered.groupby("Cluster")[TARGET_COL]
        .mean()
        .sort_values(ascending=False)
        .to_frame("Revenue_Rate")
    )

print("\n=====================================================\n")

---

## Summary of Results

| Model | Metric | Score |
|---|---|---|
| Logistic Regression | Accuracy | 0.85 |
| Logistic Regression | Recall | 0.75 |
| Logistic Regression | ROC-AUC | 0.90 |
| K-Means | Best k (Silhouette) | 2 |
| K-Means | Max Silhouette Score | 0.29 |

## Key Takeaways

- Combining supervised and unsupervised approaches gives a much richer picture than either alone
- **Prediction** enables real-time intervention at the session level
- **Clustering** enables strategic segmentation for longer-term marketing and UX decisions
- Class imbalance is a real challenge in e-commerce data — metric selection matters more than accuracy
- Logistic Regression's interpretability is valuable in business settings where explainability matters

---

*Tony James — MSc Data Science, York St John University, London*  
[LinkedIn](https://linkedin.com/in/tony-james-4bba63195) | [GitHub](https://github.com/itsmetonyjames)
